# Reduction parameters explained

This notebook explains every parameter of the STARSHIPS data reduction pipeline (`pipeline/reduction.py`, `starships.transpec.ReductionParams`), and shows their effect on a real dataset (WASP-33b, dayside/emission, SPIRou, night 1 — the same small local dataset used by the `reduction` regression tests).

In practice, only three parameters are routinely changed per dataset: `mask_tellu`, `mask_wings`, and `n_pc`. The rest (`reference_spec_box`, `reference_spec_gauss_box`, `tresh`, `tresh_lim`, `last_tresh`, `last_tresh_lim`) are effectively fixed constants of the reduction — they're documented and overridable (see `ReductionParams` / `config_dict['reduction_params']`), but you should rarely need to touch them.

This notebook only needs raw FITS files and STARSHIPS itself — no petitRADTRANS, so it runs anywhere (including locally).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yaml

import pipeline.reduction as red

CONFIG_PATH = '/Users/antoinedb/VSCodeProjects/starships_cleanup/settings/config_wasp33_day1.yaml'
VISIT_NAME = 'visit_2019-10-07'

with open(CONFIG_PATH) as f:
    config_dict = yaml.safe_load(f)

planet, obs = red.load_planet(config_dict, VISIT_NAME)
print(f"Loaded {obs.n_spec if hasattr(obs, 'n_spec') else len(obs.filenames)} exposures for {planet.name}")

## `mask_tellu` — telluric masking threshold

Fraction of transmitted flux below which a pixel is considered a *deep telluric line* and masked out entirely (rather than corrected). Typically varied between **0.2** (mask only the deepest tellurics) and **0.5** (mask more aggressively). Lower `mask_tellu` keeps more pixels but risks leaving imperfectly-corrected telluric residuals in the data; higher `mask_tellu` is safer but discards more of the spectrum.

## `mask_wings` — telluric line wing buffer

Buffer/wing limit around each masked telluric line (`lim_buffer`): pixels near a masked line, even if not deep enough to be masked on their own, are also masked up to this limit. Typically varied between **0.9** and **0.98**. A value closer to 1 masks a narrower buffer (keeps more pixels near telluric lines); a lower value masks a wider buffer.

In [ ]:
# Effect of mask_tellu / mask_wings on how much of the spectrum is masked as telluric.
n_pc_fixed = 5
for mask_tellu in [0.2, 0.35, 0.5]:
    visit = red.build_trans_spec(config_dict, n_pc_fixed, mask_tellu, 0.9, obs, planet)
    masked_fraction = visit.fl_masked.mask.mean()
    print(f"mask_tellu={mask_tellu:.2f}  ->  {masked_fraction:.1%} of pixels masked as deep telluric")

## `n_pc` — number of PCA components removed

The one reduction parameter that is essentially always tuned per dataset. After building the reference spectrum and dividing it out, STARSHIPS runs a PCA on the residual time series (across all orders jointly) to remove systematics that are common in time across many wavelengths (telluric residuals, airmass trends, instrumental drifts). `n_pc` is how many of the leading PCA components are subtracted.

- Too few components: systematics remain in the data, contaminating any later cross-correlation/retrieval.
- Too many components: risk removing part of the real planetary signal along with the systematics (the planetary signal also varies smoothly across the visit, so at high enough `n_pc` the PCA starts eating into it too).

In practice, this is tuned per dataset (typically somewhere between 1 and 12) — the plot below shows the leading PCA components' explained variance for this dataset, which gives a rough sense of where the "real systematics" stop and noise begins.

In [ ]:
visit = red.build_trans_spec(config_dict, 8, 0.2, 0.9, obs, planet)

plt.figure(figsize=(6, 4))
plt.plot(np.arange(1, 11), visit.pca.explained_variance_ratio_[:10], 'o-')
plt.xlabel('PCA component #')
plt.ylabel('Explained variance ratio')
plt.title(f'{planet.name} — leading PCA components (night {VISIT_NAME})')
plt.yscale('log')
plt.show()

## `iout_all` — which exposures build the reference spectrum

`'all'` (default, config key `iout_all`): every exposure is used to build the reference spectrum. The planetary signal is negligible next to the star's and further diluted by the planet's own motion across exposures, so using *all* exposures actually improves the reference spectrum's signal-to-noise ratio.

`null`/`None`: only the true out-of-transit/out-of-eclipse exposures (computed from the orbit) are used — the more "classical" approach, at the cost of a noisier reference spectrum (fewer exposures) if only a small fraction of the visit is genuinely out-of-transit/eclipse.

## `minimum_signal` — low-flux pixel masking

Optional: mask pixels with fewer than this many raw counts. `None` (default) disables this. Useful if some pixels should already have been masked by an earlier reduction step but weren't (e.g. detector edges).

## `bad_indexs` — exposure exclusion

Per-visit list of exposure indices to exclude entirely from the reduction (not just flagged afterwards) — e.g. exposures affected by clouds, guiding loss, etc. Set under `config_dict['bad_indexs'][visit_name]`.

## The remaining, rarely-changed parameters

See `starships.transpec.ReductionParams`'s docstring for the full detail on `reference_spec_box`, `reference_spec_gauss_box`, `tresh`, `tresh_lim`, `last_tresh`, `last_tresh_lim` — these control the smoothing kernel used to build the reference spectrum, and the sigma-clipping thresholds used to mask high-variance pixels before and after the PCA step. They're exposed and documented, but effectively fixed in practice.